# Result analysis

## Data extraction

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np


# Model keys
class MK:
    TP = "tp"
    VT = "vt"
    RX = "rx"


class Column:
    ELAPSED = "elapsed_seconds"
    CPU = "cpu_percentage"
    STACK = "stack_committed_mb"
    HEAP = "heap_used_mb"
    GCT = "gct"


thread_pool = pd.read_csv("thread-pool/cpu-memory-usage.csv", skipinitialspace=True)
virtual_thread = pd.read_csv("virtual-thread/cpu-memory-usage.csv", skipinitialspace=True)
reactive = pd.read_csv("reactive/cpu-memory-usage.csv", skipinitialspace=True)

raw_data = {
    MK.TP: {"data": thread_pool, "start_sec": 134},
    MK.VT: {"data": virtual_thread, "start_sec": 158},
    MK.RX: {"data": reactive, "start_sec": 144}
}

time_span_per_test = 600

num_tests_per_model = 4

total_time_span = time_span_per_test * num_tests_per_model

window_size_for_plot = int(time_span_per_test / 2)

window_start_for_plot = int(time_span_per_test / 4)

test_starts = [(1000, 0),
                  (500, time_span_per_test),
                  (200, time_span_per_test * 2),
                  (100, time_span_per_test * 3)]


In [ ]:
def extract_column(df, start_sec, col):
    length = total_time_span

    df = df[df[Column.ELAPSED] >= start_sec]

    time_col = df[Column.ELAPSED] - start_sec

    col = df[col]
    result = [None] * length
    for time, col_val in zip(time_col, col):
        if time < length:
            result[time] = col_val

    result = pd.Series(result).ffill().bfill().tolist()
    return result


def avg(values):
    return round(sum(values) / len(values), 1)


def span(values):
    return round(values[-1] - values[0], 6)


def extract_results(raw_data, col_name,
                 summary_avg=False, summary_change=False):
    print(f'\n--- {col_name} ---')

    col_tp = extract_column(raw_data[MK.TP]['data'], raw_data[MK.TP]['start_sec'], col_name)
    col_vt = extract_column(raw_data[MK.VT]['data'], raw_data[MK.VT]['start_sec'], col_name)
    col_rx = extract_column(raw_data[MK.RX]['data'], raw_data[MK.RX]['start_sec'], col_name)

    combined_data = {}
    for delay, start in test_starts:
        start += window_start_for_plot
        end = start + window_size_for_plot

        sample_tp = col_tp[start: end]
        sample_vt = col_vt[start: end]
        sample_rx = col_rx[start: end]

        if summary_avg:
            print(f'\nAverage:')
            print(f'{delay} ms, {MK.TP}: {avg(sample_tp)}')
            print(f'{delay} ms, {MK.VT}: {avg(sample_vt)}')
            print(f'{delay} ms, {MK.RX}: {avg(sample_rx)}')

        if summary_change:
            print(f'\nSpan:')
            print(f'{delay} ms, {MK.TP}: {span(sample_tp)}')
            print(f'{delay} ms, {MK.VT}: {span(sample_vt)}')
            print(f'{delay} ms, {MK.RX}: {span(sample_rx)}')

        combined_data[delay] = {
            MK.TP: sample_tp,
            MK.VT: sample_vt,
            MK.RX: sample_rx
        }

    return combined_data


cpu_data = extract_results(raw_data, Column.CPU, summary_avg = True)
stack_data = extract_results(raw_data, Column.STACK, summary_avg = True)
heap_data = extract_results(raw_data, Column.HEAP, summary_avg = True)

gct_data = extract_results(raw_data, Column.GCT, summary_change = True)


## CPU plot

In [ ]:
# 1. Setup the figure and grid (2 rows, 2 columns)
fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=True)
axes = axes.flatten()

# Configuration for colors and labels
delays = [100, 200, 500, 1000]
models = {
    'Thread pool': {'color': '#1f77b4', 'linestyle': '-'},
    'Virtual threads': {'color': '#2ca02c', 'linestyle': '-'},
    'Reactive streams': {'color': '#ff7f0e', 'linestyle': '--'} # Dashed to distinguish from VT
}

# 2. Loop through each delay to create a subplot
for i, delay in enumerate(delays):
    ax = axes[i]

    # --- DATA PLUG-IN START ---
    time_sec = np.arange(window_size_for_plot)

    # Mock data for demonstration - replace with your actual CPU records
    tp_data = cpu_data[delay][MK.TP]
    vt_data = cpu_data[delay][MK.VT]
    rx_data = cpu_data[delay][MK.RX]

    ax.plot(time_sec, tp_data, label='Thread pool', **models['Thread pool'])
    ax.plot(time_sec, vt_data, label='Virtual threads', **models['Virtual threads'])
    ax.plot(time_sec, rx_data, label='Reactive streams', **models['Reactive streams'])
    # --- DATA PLUG-IN END ---

    # Subplot Styling
    ax.set_title(f'Simulated delay: {delay}ms', fontsize=12, fontweight='bold')
    ax.grid(True, linestyle=':', alpha=0.6)

    # Axis Labels (Only on the outer edges for cleanliness)
    if i >= 2:
        ax.set_xlabel('Time (seconds)', fontsize=11)
    if i % 2 == 0:
        ax.set_ylabel('CPU usage (%)', fontsize=11)

# 3. Global Figure Styling
plt.suptitle('CPU utilization across JVM concurrency models under variable I/O latency',
             fontsize=16, fontweight='bold', y=0.98)

# Create a single legend at the top
handles, labels = ax.get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 0.94),
           ncol=3, frameon=False, fontsize=12)

# Adjust layout to make room for titles and legend
plt.tight_layout(rect=[0, 0.03, 1, 0.93])

# 4. Save and Show
plt.savefig('cpu_concurrency_grid.png', dpi=300, bbox_inches='tight')
plt.show()

## Memory plot

In [ ]:
def draw_memory_plot(delays, figsize):
    # Set figure size for a vertical layout
    fig, axes = plt.subplots(len(delays), 2, figsize=figsize, sharex=True)

    colors = {'TP': '#1f77b4', 'VT': '#2ca02c', 'RX': '#ff7f0e'}
    labels = {'TP': 'Thread Pool', 'VT': 'Virtual Threads', 'RX': 'Reactive Streams'}

    for i, delay in enumerate(delays):
        # Left Column: Heap Used
        ax_h = axes[i, 0]
        ax_h.plot(time_sec, heap_data[delay][MK.TP], color=colors['TP'], label=labels['TP']) # Replace with actual data
        ax_h.plot(time_sec, heap_data[delay][MK.VT], color=colors['VT'], label=labels['VT'])
        ax_h.plot(time_sec, heap_data[delay][MK.RX], color=colors['RX'], label=labels['RX'], ls='--')

        ax_h.set_title(f'Heap Utilization: {delay}ms Delay', fontweight='bold')
        ax_h.set_ylabel('MB')
        ax_h.grid(True, alpha=0.3)

        # Right Column: Stack Reserved
        ax_s = axes[i, 1]
        ax_s.plot(time_sec, stack_data[delay][MK.TP], color=colors['TP'])
        ax_s.plot(time_sec, stack_data[delay][MK.VT], color=colors['VT'])
        ax_s.plot(time_sec, stack_data[delay][MK.RX], color=colors['RX'], ls='--')

        ax_s.set_title(f'Stack Reservation: {delay}ms Delay', fontweight='bold')
        ax_s.set_ylabel('MB')
        ax_s.grid(True, alpha=0.3)

        if i == 3: # X-labels only for bottom row
            ax_h.set_xlabel('Time (Seconds)')
            ax_s.set_xlabel('Time (Seconds)')

    # Global Legend
    handles, labs = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labs, loc='upper center', bbox_to_anchor=(0.5, 0.99),
               ncol=3, frameon=False, fontsize=12)

    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.savefig(f'memory_metrics_{'_'.join([str(x) for x in delays])}.png', dpi=300)
    plt.show()


# delays = [100, 200, 500, 1000]
draw_memory_plot([100, 200], (12, 9))
draw_memory_plot([500, 1000], (12, 9))
